# Задание 3. Восстановление трёх фильтров KaiNet

Даны два фильтра 3×3 и 1000 пар входных и выходных матриц. Требуется восстановить третий фильтр и порядок применения.

## Метод
Обозначим строки `algos.csv` как A и B, неизвестный фильтр — X. Для каждого из шести порядков перебираем пять способов обработки границ. При фиксированных A, B и порядке весь результат линейно зависит от девяти коэффициентов X.

По очереди подставляем девять базисных фильтров (одна единица, остальные нули). Полученные выходы образуют столбцы матрицы D. Решаем D @ x = y методом наименьших квадратов `numpy.linalg.lstsq`. Для восстановления используем только первые пять изображений; затем проверяем результат на всех 1000, включая 995 не использованных при восстановлении.

Используется операция, описанная в условии: скользящее поэлементное произведение без переворота ядра (`scipy.ndimage.correlate`). Найденная обработка границ — дополнение нулями на один пиксель перед каждым из трёх фильтров. Изображения обрабатываются как float64 в исходном диапазоне 0–255.

## Воспроизведение
Нужны numpy, scipy и Pillow. Папка `data` должна содержать распакованный архив. Запустите следующую ячейку из папки проекта или `task3`. Скрипт восстанавливает фильтр, проверяет результат и сохраняет CSV и подробный отчёт.


In [1]:
from pathlib import Path
import runpy

folder = Path.cwd() if Path('reconstruct.py').exists() else Path.cwd() / 'task3'
runpy.run_path(str(folder / 'reconstruct.py'), run_name='__main__');

{
  "missing_filter": [
    [
      0.125,
      0.25,
      0.125
    ],
    [
      0.25,
      0.5,
      0.25
    ],
    [
      0.125,
      0.25,
      0.125
    ]
  ],
  "validation": {
    "XBA": {
      "images": 1000,
      "pixels": 4096000,
      "mse": 0.0,
      "max_abs_error": 0.0
    },
    "BXA": {
      "images": 1000,
      "pixels": 4096000,
      "mse": 0.0,
      "max_abs_error": 0.0
    }
  },
  "note": "XBA and BXA are observationally indistinguishable; original first two positions are not identifiable."
}


## Результат и неоднозначность порядка

Восстановлен фильтр:

```
0.125  0.25  0.125
0.25   0.5   0.25
0.125  0.25  0.125
```

Матрица системы имеет ранг 9. Для порядков **X → B → A** и **B → X → A** MSE выходов и максимальная абсолютная ошибка равны **нулю** на всех 1000 изображениях (4 096 000 пикселей). Остальные порядки и проверенные способы обработки границ точного совпадения не дают.

Эта неоднозначность не является погрешностью подбора. B — разделимый фильтр с одномерными весами [1, 1, 1], X — с весами [1, 2, 1]. При нулевых границах соответствующие одномерные операторы имеют вид T и T + I, поэтому коммутируют. Двумерные разделимые операторы также коммутируют. Следовательно, никакие дополнительные входные изображения не позволят различить эти два порядка в данной модели вычислений.

`reconstructed_algos.csv` содержит X, B, A; `reconstructed_algos_alternative.csv` содержит B, X, A. Предпочтение первого варианта условное. Поскольку условие говорит об MSE **самих фильтров**, а не выходов, проверяющая система может различать эти два эквивалентных решения. Правильный исходный порядок первых двух фильтров можно уточнить по результату контеста. Локальная проверка не подтверждает принятие ответа в контесте.


## Результат контеста
Основной `reconstructed_algos.csv` принят 28 сентября 2026: **OK, полное решение**. [Посылка №166717004](https://contest.yandex.ru/contest/75233/run-report/166717004/).
